<!--
Copyright (c) 2022-2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
See LICENSE for license information.
-->

# Linear Attention

Linear attention maintains a recurrent state instead of forming the pairwise attention scores used by scaled dot product attention (SDPA). Transformer Engine provides PyTorch modules for several linear attention variants through the cuDNN frontend. These APIs are **experimental** and subject to change.

This guide introduces the available variants and shows Gated DeltaNet (GDN) forward and backward on a single GPU and with context parallelism (CP). For SDPA backends, masks, and biases, see [Attention Is All You Need!](attention.ipynb).


## 1. Supported Variants

| Variant | PyTorch API | Gates and updates | Context parallelism |
| --- | --- | --- | --- |
| Gated DeltaNet (GDN) | [GatedDeltaNetAttention](../../api/pytorch.rst#transformer_engine.pytorch.GatedDeltaNetAttention) | Per-token, per-head log decay `g` and write strength `beta` | Contiguous CP |
| Gated DeltaNet v2 (GDN-2) | [GatedDeltaNet2Attention](../../api/pytorch.rst#transformer_engine.pytorch.GatedDeltaNet2Attention) | Per-key-channel `g` and `beta`, per-value-channel `w` | Not supported |
| Gated DeltaProduct (GDP) | [GatedDeltaProductAttention](../../api/pytorch.rst#transformer_engine.pytorch.GatedDeltaProductAttention) | Multiple delta-rule updates per token | Not supported |
| Kimi Delta Attention (KDA) | [KimiDeltaAttention](../../api/pytorch.rst#transformer_engine.pytorch.KimiDeltaAttention) | Per-key-channel decay `g`, per-token, per-head `beta` | Not supported |

The modules take Q, K, V and gate tensors produced by the caller; they do not include input projections or an output projection. Q/K/V use FP16 or BF16, and recurrent states use FP32. Gate shapes and dtypes depend on the variant; see the corresponding API reference.

Install Transformer Engine with PyTorch support as described in [Installation](../../installation.rst). The cuDNN frontend must provide the selected linear attention op and its kernel runtime. See the [cuDNN frontend installation guidance](https://github.com/NVIDIA/cudnn-frontend#installation) for its dependencies and installation options. Supported GPU architectures and head dimensions depend on the variant and the available cuDNN frontend engines.


## 2. GDN Inputs and Outputs

GDN is causal; its API does not take SDPA attention masks or bias tensors. It takes two FP32 gates: `g`, the log decay, and `beta`, the write strength. Each gate has one value per token and head. The example uses negative `g` and `beta` between zero and one, and enables Q/K L2 normalization with `use_qk_l2norm_in_kernel=True`.

| `qkv_format` | Q/K shape | V shape | Gate shape | Output shape |
| --- | --- | --- | --- | --- |
| `bshd` | `[B, S, H, Dk]` | `[B, S, H, Dv]` | `[B, S, H]` | `[B, S, H * Dv]` |
| `sbhd` | `[S, B, H, Dk]` | `[S, B, H, Dv]` | `[S, B, H]` | `[S, B, H * Dv]` |
| `thd` | `[T, H, Dk]` | `[T, H, Dv]` | `[T, H]` | `[T, H * Dv]` |

Here `B` is the number of documents, `S` is the dense sequence length, `T` is the total number of packed tokens, and `H` is the number of heads on this tensor-parallel rank. `kv_channels` sets `Dk` and `Dv`: pass one integer for equal head dimensions or `(Dk, Dv)` otherwise.

For packed inputs, remove padding and pass `cu_seqlens`, the cumulative document lengths, as an `int32` tensor. Documents do not share recurrent state. Without CP, `cu_seqlens` must be on the input device. Dense inputs can omit it; if supplied, it must describe full rows as `[0, S, 2*S, ..., B*S]`.

The optional `initial_state` has shape `[B, H, Dv, Dk]` and dtype FP32. Omitting it starts from zero. Set `output_final_state=True` to return `(output, final_state)`; otherwise the module returns only `output`. Final states have the same shape and dtype as the initial state and can seed a subsequent call.


## 3. Single-GPU Example

The runnable example below uses 16 documents and 8 heads with BF16 Q/K/V and FP32 gates and states. Dense sequences have 128 tokens each; packed document lengths are sampled from 32 to 128 tokens.

Run these commands from the repository root:

```bash
python docs/examples/attention/example_linear_attention.py
python docs/examples/attention/example_linear_attention.py --qkv-format thd
```

Each run checks output and state shapes and finite gradients after forward and backward. Missing kernel dependencies fail the run.

The input preparation covers equal-length dense rows and packed documents of different lengths:

The forward call below works with either layout. Backward includes a loss on both the attention output and the final state:

## 4. GDN Context Parallelism

GDN supports **contiguous context parallelism**. Create a `torch.distributed` process group and pass it as `cp_group` when constructing the module, or call `set_context_parallel_group` before forward. `None` disables CP; a group of size one follows the local path.

Each rank supplies local Q/K/V and gate tensors. Group rank order defines token order, and all inputs must use the same partition:

- **THD:** rank `r` owns `[r*T//P:(r+1)*T//P]` of the global packed buffer.
- **BSHD/SBHD:** rank `r` owns `[r*S//P:(r+1)*S//P]` of every batch row.

`P` is the CP group size. These are contiguous intervals, rather than SDPA's dual-chunk partitioning. TE does not redistribute inputs, and outputs stay on the ranks that own their input tokens.

Pass the same **global** `cu_seqlens` on every rank, describing the complete documents before partitioning. For THD it is required. Dense inputs can omit it for equal slices; uneven dense slices require `[0, S, 2*S, ..., B*S]`. With CP, this metadata may be on CPU or CUDA. Prefer CPU metadata when already available from the input pipeline to avoid a synchronized host read.

Document lengths need not be divisible by CP size. Empty documents and ranks with no tokens are supported. All ranks must participate in forward and backward with matching options, including ranks with no tokens or no local loss.

The example selects the local intervals as follows (the `bshd` path slices each batch row):

Launch the dense or packed example on two GPUs:

```bash
python -m torch.distributed.run --standalone --nproc-per-node=2 \
    docs/examples/attention/example_linear_attention.py --cp

python -m torch.distributed.run --standalone --nproc-per-node=2 \
    docs/examples/attention/example_linear_attention.py --cp --qkv-format thd
```

Each run compares local outputs, replicated final states, and input and initial-state gradients against a non-CP run on the complete documents.

For CP, the script selects NCCL, [PyTorch's recommended backend for CUDA distributed training](https://docs.pytorch.org/docs/stable/distributed.html#which-backend-to-use), and selects each process's GPU using `LOCAL_RANK`. GDN CP also accepts other process-group backends that support its CUDA collectives. For example, a CUDA-capable Gloo build can use `backend="gloo"` in `init_process_group`. For demonstration, each rank generates the same full batch and selects its interval locally. In a training pipeline, supply the local tensors directly.

This is the CP forward and backward:

### Recurrent States and Losses

With CP, `initial_state` describes **all global documents** and must have identical values on every rank. Requested final states are also replicated for all global documents. Empty documents retain their initial state.

Each rank computes the loss for its own output tokens. Dividing each local sum by the total number of output elements across the CP group makes the combined loss equal to the non-CP mean. Final states are copied to every rank, and their gradients are summed during backward. If every rank applies the same final-state loss, divide it by CP size so it is counted only once.

Each document's initial-state gradient is returned on its first owning rank, with zeros elsewhere. Sum these contributions over CP when inspecting a replicated leaf state's full gradient. Final states can also be chained into subsequent CP calls.

### Supported Execution

GDN CP supports FP16/BF16 inputs, FP32 gates and states, Q/K normalization, activation checkpointing (`checkpoint_core_attention=True`), and tensor parallelism. It supports first-order gradients and executes eagerly, including inside a `torch.compile` model. Full-graph compilation and CUDA graph capture are outside the supported CP execution modes. Without CP, ordinary forward/backward calls support `torch.compile(fullgraph=True)`; activation checkpointing still requires eager attention execution.